# Why pangolin?

This notebook exhibits the advantages of Pangolin over NumPyro

Initial advantages

1. Simple, minimal API
2. No hideous strings

In [1]:
import pangolin
from pangolin import interface as pi
import numpyro
from numpyro import distributions as ndist
import jax
from jax import numpy as jnp
import numpy as np
import functools

## No string labels + clear inspectable IR

In pangolin, a model is just a DAG of random variables. You can easily inspect it or plot it.

In NumPyro, a model is a traced function. You can only inspect by calling it.

In [138]:
# pangolin

x = [pi.normal(0, 1)]
for i in range(1,4):
    x.append(pi.normal(x[-1], i*10))
y = pi.normal(x[-1],42)

print(f"{y.parents[0] == x[-1] = }")

pangolin.ir.graph_upstream(y)

y.parents[0] == x[-1] = True
○             () 0
│
│ ○           () 1
│ │
│ │ ○         () 10
│ │ │
│ │ │ ○       () 20
│ │ │ │
│ │ │ │ ○     () 30
│ │ │ │ │
│ │ │ │ │ ○   () 42
│ │ │ │ │ │
└─○ │ │ │ │   () normal
  │ │ │ │ │
  └─○ │ │ │   () normal
    │ │ │ │
    └─○ │ │   () normal
      │ │ │
      └─○ │   () normal
        │ │
        └─○   () normal


In [133]:
# numpyro

def model(y=None):
    x_prev = numpyro.sample("x_0", ndist.Normal(0, 1))
    for i in range(1, 4):
        x_prev = numpyro.sample(f"x_{i}", ndist.Normal(x_prev, i*10))
    return numpyro.sample("y", ndist.Normal(x_prev, 5.5), obs=y)

y_obs = -4.0

In [124]:
# In theory this should show a graph, of random vars only, if you install graphviz, but doesn't support scan

#numpyro.render_model(model, model_args=(y_obs,), render_distributions=True)

In [125]:
# how to even inspect this? you can do a trace

exec_trace = numpyro.handlers.trace(numpyro.handlers.seed(model, rng_seed=0)).get_trace(y_obs)
print(exec_trace)

OrderedDict({'x_0': {'type': 'sample', 'name': 'x_0', 'fn': <numpyro.distributions.continuous.Normal object at 0x7feda5728520 with batch shape () and event shape ()>, 'args': (), 'kwargs': {'rng_key': Array((), dtype=key<fry>) overlaying:
[ 928981903 3453687069], 'sample_shape': ()}, 'value': Array(-2.4424558, dtype=float32), 'scale': None, 'is_observed': False, 'intermediates': [], 'cond_indep_stack': [], 'infer': {}}, 'x_1': {'type': 'sample', 'name': 'x_1', 'fn': <numpyro.distributions.continuous.Normal object at 0x7feda5728590 with batch shape () and event shape ()>, 'args': (), 'kwargs': {'rng_key': Array((), dtype=key<fry>) overlaying:
[1353695780 2116000888], 'sample_shape': ()}, 'value': Array(-15.017233, dtype=float32), 'scale': None, 'is_observed': False, 'intermediates': [], 'cond_indep_stack': [], 'infer': {}}, 'x_2': {'type': 'sample', 'name': 'x_2', 'fn': <numpyro.distributions.continuous.Normal object at 0x7feda5728910 with batch shape () and event shape ()>, 'args': (),

In [126]:
# you can get model relations
numpyro.infer.inspect.get_model_relations(model, model_args=(y_obs,))

{'sample_sample': {'x_0': [],
  'x_1': ['x_0'],
  'x_2': ['x_1'],
  'x_3': ['x_2'],
  'y': ['x_3']},
 'sample_param': {'x_0': [], 'x_1': [], 'x_2': [], 'x_3': [], 'y': []},
 'sample_dist': {'x_0': 'Normal',
  'x_1': 'Normal',
  'x_2': 'Normal',
  'x_3': 'Normal',
  'y': 'Normal'},
 'param_constraint': {},
 'plate_sample': {},
 'observed': ['y']}

In [131]:
# you can turn it into a density and then evaluate it
params = {f"x_{i}":0.0 for i in range(5)}
ld = functools.partial(numpyro.infer.util.log_density, model, (y_obs,), {})
f = lambda params: ld(params)[1]
f(params)

OrderedDict([('x_0',
              {'type': 'sample',
               'name': 'x_0',
               'fn': <numpyro.distributions.continuous.Normal object at 0x7feda5ae6dd0 with batch shape () and event shape ()>,
               'args': (),
               'kwargs': {'rng_key': None, 'sample_shape': ()},
               'value': 0.0,
               'scale': None,
               'is_observed': False,
               'intermediates': [],
               'cond_indep_stack': [],
               'infer': {}}),
             ('x_1',
              {'type': 'sample',
               'name': 'x_1',
               'fn': <numpyro.distributions.continuous.Normal object at 0x7feda5ae6ac0 with batch shape () and event shape ()>,
               'args': (),
               'kwargs': {'rng_key': None, 'sample_shape': ()},
               'value': 0.0,
               'scale': None,
               'is_observed': False,
               'intermediates': [],
               'cond_indep_stack': [],
               'infer'

In [122]:
# in principle you should be able to look at the jaxpr, but this doesn't work because of
# proprietary NumPyro confusion

#jax.make_jaxpr(f)(params) # doesn't work, who knows why

## Sane vectorization

Consider a mathematical model of the form

$$a_i \sim \text{normal}(0,10)$$

$$b_{jk} \sim \text{normal}(0,5)$$

$$c_{ik} \sim \text{lognormal}(0,1)$$

$$z_{ijk} \sim \text{normal}(a_i + b_{jk} x_{ik}, c_{ik})$$

In pangolin, there's a direct mathematical translation and you can easily check that the output is what you intend.

In [91]:
I = 10
J = 20
K = 30

x = np.random.randn(I,K)
x_rv = pi.constant(x)

a = pi.vfor(lambda i: pi.normal(0, 10), i=I)
b = pi.vfor(lambda j, k: pi.normal(0, 5), j=J, k=K)
c = pi.vfor(lambda i, j: pi.lognormal(0, 1), i=I, j=J)
z = pi.vfor(lambda i, j, k: pi.normal(a[i] + b[j,k] * x_rv[i,k], c[i, j]), i=I, j=J, k=K)

print(f"{a.shape=} = (I,)")
print(f"{b.shape=} = (J, K)")
print(f"{c.shape=} = (I, J)")
print(f"{z.shape=} = (I, J, K)")

a.shape=(10,) = (I,)
b.shape=(20, 30) = (J, K)
c.shape=(10, 20) = (I, J)
z.shape=(10, 20, 30) = (I, J, K)


In NumPyro you have to deal with normal NumPy broadcasting insanity

In [92]:
def model(x, z=None):
    a = numpyro.sample("a", ndist.Normal(0, 10).expand([I]))
    b = numpyro.sample("b", ndist.Normal(0, 5).expand([J, K]))
    c = numpyro.sample("c", ndist.LogNormal(0, 1).expand([I, J]))
    mean = a[None, None, :] + b[:, None, :] * x[None, :, :]
    z = numpyro.sample("z", ndist.Normal(mean, c[None, :, :]), obs=z)

Not bad, right? Except... that code contains four bugs. Can you find them?

It only looks simple because your eyes glazed over all the dimension-mashing. Here's the *actual* NumPyro code.

In [93]:
def model(x, z=None):
    a = numpyro.sample("a", ndist.Normal(0, 10).expand([I]))
    b = numpyro.sample("b", ndist.Normal(0, 5).expand([J, K]))
    c = numpyro.sample("c", ndist.LogNormal(0, 1).expand([I, J]))
    mean = a[:, None, None] + b[None, :, :] * x[:, None, :]
    z = numpyro.sample("z", ndist.Normal(mean, c[:, :, None]), obs=z)

and here's how you could verify that the shapes are as you expect

In [64]:
x = jnp.array(np.random.randn(I,K))
tr = numpyro.handlers.trace(numpyro.handlers.seed(model, rng_seed=0)).get_trace(x)

for name in ["a", "b", "c", "z"]:
    site = tr[name]
    print(name, site["value"].shape)

a (10,)
b (20, 30)
c (10, 20)
z (10, 20, 30)


But doesn't NumPyro have `plate`? Sure, but it doesn't help:

In [106]:
def model(x, z=None):
    with numpyro.plate("i", I, dim=-1):
        a = numpyro.sample("a", ndist.Normal(0, 10))

    with numpyro.plate("j", J, dim=-2):
        with numpyro.plate("k", K, dim=-1):
            b = numpyro.sample("b", ndist.Normal(0, 5))

    with numpyro.plate("i", I, dim=-2):
        with numpyro.plate("j", J, dim=-1):
            c = numpyro.sample("c", ndist.LogNormal(0, 1))

    mean = a[:, None, None] + b[None, :, :] * x[:, None, :]
    # mean = a[:, None, None] + jnp.einsum("jk,ik->ijk", b, x) # or you could do this??

    with numpyro.plate("i", I, dim=-3):
        with numpyro.plate("j", J, dim=-2):
            with numpyro.plate("k", K, dim=-1):
                numpyro.sample("z", ndist.Normal(mean, c[:, :, None]), obs=z)


In [107]:
x = jnp.array(np.random.randn(I,K))
tr = numpyro.handlers.trace(numpyro.handlers.seed(model, rng_seed=0)).get_trace(x)

for name in ["a", "b", "c", "z"]:
    site = tr[name]
    print(name, site["value"].shape)

a (10,)
b (20, 30)
c (10, 20)
z (10, 20, 30)


## Automatic deferred inference

Take the model

In [167]:
# Pangolin
x_train = pi.constant(np.random.randn(100))
y_obs   = np.random.randn(100)
x_test  = pi.constant(np.random.randn(10000))

alpha, beta = pi.normal(0, 10), pi.normal(0, 10)
sigma    = pi.lognormal(0, 1)
y_train  = pi.vfor(lambda i: pi.normal(alpha * beta*x_train[i], sigma))
y_test   = pi.vfor(lambda i: pi.normal(alpha * beta*x_test[i], sigma))

y_pred = pangolin.blackjax.E(y_test, y_train, y_obs)
print(y_pred.shape)


(10000,)


In [170]:
# NumPyro — the documented workflow
def model(x, y=None):
    alpha = numpyro.sample("alpha", ndist.Normal(0, 10))
    beta  = numpyro.sample("beta",  ndist.Normal(0, 10))
    sigma = numpyro.sample("sigma", ndist.LogNormal(0, 1))
    numpyro.sample("y_train", ndist.Normal(alpha + beta * x_train.op.value, sigma), obs=y)
    numpyro.sample("y_test", ndist.Normal(alpha + beta * x_test.op.value, sigma))

key = jax.random.PRNGKey(0)
mcmc = numpyro.infer.MCMC(numpyro.infer.NUTS(model), num_warmup=500, num_samples=1000)
mcmc.run(key, x=x_train.op.value, y=y_obs)

# predictive = numpyro.infer.Predictive(model, posterior_samples=mcmc.get_samples())
# y_test_samples = predictive(key, x=x_test.op.value, y=None)["y"]
# y_pred = jnp.mean(y_test_samples,axis=0)
# print(y_pred.shape)

sample: 100%|██████████| 1500/1500 [00:26<00:00, 56.89it/s, 63 steps of size 9.68e-02. acc. prob=0.89]


In [ ]:
# NumPyro — the documented workflow
def model(x, y=None):
    alpha = numpyro.sample("alpha", ndist.Normal(0, 10))
    beta  = numpyro.sample("beta",  ndist.Normal(0, 10))
    sigma = numpyro.sample("sigma", ndist.LogNormal(0, 1))
    numpyro.sample("y", ndist.Normal(alpha + beta * x, sigma), obs=y)

key = jax.random.PRNGKey(0)
mcmc = numpyro.infer.MCMC(numpyro.infer.NUTS(model), num_warmup=500, num_samples=1000)
mcmc.run(key, x=x_train.op.value, y=y_obs)

predictive = numpyro.infer.Predictive(model, posterior_samples=mcmc.get_samples())
y_test_samples = predictive(key, x=x_test.op.value, y=None)["y"]
y_pred = jnp.mean(y_test_samples,axis=0)
print(y_pred.shape)

sample: 100%|██████████| 1500/1500 [00:01<00:00, 1439.02it/s, 1 steps of size 6.77e-01. acc. prob=0.94]


(10000,)
